# 02 — Exploratory analysis of article metadata and text

This notebook loads a **local** SQLite database, checks data quality, calculates descriptive statistics and saves figures. It does not download data. Place your authorized database at `data/ScrapedNiusData.db` locally; do not commit copyrighted article text.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = Path("../data/ScrapedNiusData.db")
FIGURES = Path("../outputs/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH.resolve()}. Add an authorized local copy first.")

with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
    print("Tables:", tables['name'].tolist())
    table_name = "articles" if "articles" in tables['name'].tolist() else "NiusData"
    df = pd.read_sql_query(f'SELECT * FROM "{table_name}"', conn)

# Normalize column names from either the refactored or original coursework schema.
rename = {"date": "published_at", "text": "article_text", "highlight": "highlights"}
df = df.rename(columns={k:v for k,v in rename.items() if k in df.columns and v not in df.columns})
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)


## 1. Data quality

Duplicate rows and missing fields can change the interpretation of summary statistics. Review these checks before interpreting results.

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str),
})
display(quality)

if "title" in df:
    print("Duplicate titles:", int(df["title"].duplicated().sum()))
if "article_text" in df:
    df["word_count"] = df["article_text"].fillna("").astype(str).str.split().str.len()
    display(df["word_count"].describe(percentiles=[.25, .5, .75]).to_frame())


## 2. Articles by category and author

In [ ]:
if "category" in df:
    category_counts = df["category"].fillna("Unknown").value_counts().sort_values()
    ax = category_counts.plot(kind="barh", figsize=(9, 5), title="Articles by category")
    ax.set_xlabel("Number of articles"); ax.set_ylabel("Category")
    plt.tight_layout(); plt.savefig(FIGURES / "articles_by_category.png", dpi=160); plt.show()

if "author" in df:
    author_counts = df["author"].fillna("Unknown").value_counts().head(15).sort_values()
    ax = author_counts.plot(kind="barh", figsize=(9, 6), title="Top authors by article count")
    ax.set_xlabel("Number of articles"); ax.set_ylabel("Author")
    plt.tight_layout(); plt.savefig(FIGURES / "articles_by_author.png", dpi=160); plt.show()


## 3. Article length by category

In [ ]:
if "word_count" not in df and "article_text" in df:
    df["word_count"] = df["article_text"].fillna("").astype(str).str.split().str.len()
if {"category", "word_count"}.issubset(df.columns):
    length_summary = df.groupby("category", dropna=False)["word_count"].agg(
        articles="count", mean_words="mean", median_words="median", std_words="std"
    ).sort_values("median_words", ascending=False).round(1)
    display(length_summary)
    ax = df.boxplot(column="word_count", by="category", figsize=(11, 6), rot=45, grid=False)
    ax.set_title("Article length by category"); ax.set_xlabel("Category"); ax.set_ylabel("Words")
    plt.suptitle(""); plt.tight_layout(); plt.savefig(FIGURES / "word_count_by_category.png", dpi=160); plt.show()


## 4. Publication timeline

Date parsing is intentionally tolerant because website date formats can change. Inspect invalid dates rather than silently treating them as valid observations.

In [ ]:
date_col = "published_at" if "published_at" in df else ("date" if "date" in df else None)
if date_col:
    df["published_at_parsed"] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=True)
    print("Unparsed dates:", int(df["published_at_parsed"].isna().sum()))
    monthly = df.dropna(subset=["published_at_parsed"]).set_index("published_at_parsed").resample("MS").size()
    if not monthly.empty:
        ax = monthly.plot(kind="bar", figsize=(10, 4), title="Articles by publication month")
        ax.set_xlabel("Month"); ax.set_ylabel("Number of articles")
        plt.tight_layout(); plt.savefig(FIGURES / "articles_by_month.png", dpi=160); plt.show()


## 5. Person-name extraction (optional)

Named-entity recognition is imperfect, especially for German names, quotations and short texts. Treat extracted names as candidates and validate them before drawing conclusions. Install the German spaCy model separately if you enable this section.

In [ ]:
# Optional setup (run once in your environment):
# python -m spacy download de_core_news_sm
# import spacy
# nlp = spacy.load("de_core_news_sm")
#
# def people_in_text(text):
#     if not isinstance(text, str) or not text.strip():
#         return []
#     doc = nlp(text)
#     return [ent.text for ent in doc.ents if ent.label_ in {"PER", "PERSON"}]
#
# if "article_text" in df:
#     df["person_candidates"] = df["article_text"].apply(people_in_text)


## 6. Export summary tables

Only aggregate tables and figures should be committed after checking that they do not expose personal or copyrighted content.

In [ ]:
(Path("../outputs/tables")).mkdir(parents=True, exist_ok=True)
if "category" in df:
    df["category"].fillna("Unknown").value_counts().rename_axis("category").reset_index(name="article_count").to_csv("../outputs/tables/category_counts.csv", index=False)
if "word_count" in df and "category" in df:
    df.groupby("category")["word_count"].agg(["count", "mean", "median"]).round(1).to_csv("../outputs/tables/word_count_summary.csv")
print("Aggregate tables exported.")
